# M6 — Long-run investment effectiveness

Describe historical rolling 3- and 5-year outcomes for Cameco, a fixed uranium basket, URA and SPY. This is a historical outcome study rather than a future price forecast. It uses USD returns and US CPI-U (CPIAUCNS).

Run cells in order with the project environment (`uv run jupyter lab`). Paths support the project root or `notebooks/`. See [workshop order](../_docs/workshop_order.md) for dependencies and checkpoints. Numerical outputs come from actual local inputs; no synthetic data are substituted for market results.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "config/project.yaml").exists():
    ROOT = ROOT.parent
assert (ROOT / "config/project.yaml").exists(), "Launch from the project root or notebooks/"
from uranium_signals.config import load_config
from uranium_signals.universe import load_universe
from uranium_signals.providers import acquire
from uranium_signals.run import research_tables
cfg = load_config(ROOT / "config/project.yaml")
universe = load_universe(cfg)
tickers = set(universe.ticker) | {cfg["benchmark"], cfg["broad_benchmark"]} | set(cfg["optional_benchmarks"])


In [ ]:
prices, macro, source_metadata = acquire(cfg, tickers, offline=True)
prices, quality, targets, table = research_tables(cfg, prices, macro, universe)


## 1. Define the basket and common history

The basket is fixed in configuration before observing results: CCJ, DNN, UEC, UUUU, URG and NXE. Start when all six have monthly data, hold equal weights initially, and rebalance at each January month-end after that month's return. Report this descriptive basket before transaction costs and tax. Other series retain their own available histories; matched windows support comparisons.

In [ ]:
from uranium_signals.long_horizon import analyze_long_horizon
outcomes, long_summary, levels = analyze_long_horizon(prices, macro, cfg)
print("Fixed basket:", cfg["basket"])
print("Basket common inception:", levels.BASKET.first_valid_index())
display(levels.tail())


## 2. Calculate nominal and real outcomes

Nominal holding return is ending total-return proxy divided by starting value minus one. Real return divides the gross nominal return by the CPI ratio. CAGR raises gross return to `12 / horizon_months`. Using the same currency for assets and inflation is essential.

In [ ]:
row = outcomes[outcomes.ticker.eq("CCJ") & outcomes.horizon_months.eq(36)].iloc[0]
start = pd.Period(row.start, "M"); end = pd.Period(row.end, "M")
p0, p1 = levels.loc[start, "CCJ"], levels.loc[end, "CCJ"]
cpi_frame = macro[macro.series.eq("CPIAUCNS")].copy()
cpi_frame.index = cpi_frame.date.dt.to_period("M")
cpi0, cpi1 = cpi_frame.loc[start, "value"], cpi_frame.loc[end, "value"]
nominal_return = p1 / p0 - 1
real_return = (1 + nominal_return) / (cpi1 / cpi0) - 1
real_cagr = (1 + real_return) ** (1 / 3) - 1
assert np.isclose(nominal_return, row.nominal_return)
assert np.isclose(real_return, row.real_return)
assert np.isclose(real_cagr, row.real_cagr)
print("Worked window:", start, end, "nominal", nominal_return, "real", real_return, "real CAGR", real_cagr)


## 3. Compare return, risk and effective sample size

Maximum drawdown is measured from the highest prior monthly close within each holding window; intramonth losses can be worse. Consecutive rolling outcomes share almost all observations. Non-overlapping counts show how few distinct holding intervals exist, although even non-overlapping periods need not be statistically independent.

In [ ]:
display(long_summary)
fig, axes = plt.subplots(2, 1, figsize=(11, 8))
for axis, horizon in zip(axes, (36, 60)):
    for ticker, g in outcomes[outcomes.horizon_months.eq(horizon)].groupby("ticker"):
        axis.plot(pd.to_datetime(g.start), g.real_cagr, label=ticker)
    axis.axhline(0, color="grey", linewidth=.5)
    axis.set_title(f"{horizon // 12}-year real CAGR by investment month")
    axis.legend()
plt.tight_layout(); plt.show()
assert (long_summary.nonoverlapping_windows <= long_summary.rolling_windows).all()


## Written-result checkpoint

Report nominal and real CAGR, positive-real-return frequency, benchmark-relative holding returns, monthly drawdowns, and both rolling/non-overlapping window counts. Discuss differing inception dates and the limited number of uranium cycles. No long-horizon ML model is fitted; no random split of overlapping windows is performed. Cash/bill comparison is deferred in this first version.